# 📊 Phân tích Kết quả Huấn luyện — Tuần 4

**Phụ trách:** Phạm Văn Tưởng  
**Mục tiêu:** Visualize Loss và Accuracy của ResNet50 và MobileNetV3-Large

Notebook này:
1. Đọc log CSV từ `models/classification/`
2. Vẽ Train Loss vs Val Loss theo epoch
3. Vẽ Train Accuracy vs Val Accuracy
4. So sánh 2 mô hình trên cùng biểu đồ
5. Export figure ra `figures/`

In [ ]:
import sys
import os

# Thêm thư mục gốc dự án vào PATH
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)
os.chdir(project_root)

print(f'Working directory: {os.getcwd()}')

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)
print('✅ Setup hoàn thành!')

## 1. Đọc Log CSV

In [ ]:
LOG_DIR = Path('models/classification')

def load_training_log(model_name: str) -> pd.DataFrame | None:
    """Đọc file CSV log huấn luyện."""
    log_path = LOG_DIR / f'{model_name}_training_log.csv'
    if not log_path.exists():
        print(f'[WARN] Không tìm thấy: {log_path}')
        return None
    df = pd.read_csv(log_path)
    print(f'[{model_name}] {len(df)} epochs | '
          f'best val_acc={df.val_acc.max():.2f}% (epoch {df.val_acc.idxmax()+1})')
    return df

df_resnet = load_training_log('resnet50')
df_mobile = load_training_log('mobilenetv3')

# Nếu chưa có file thực, dùng dữ liệu giả lập để demo notebook
def make_demo_history(epochs=50, model='resnet50'):
    """Tạo history giả lập cho demo."""
    rng = np.random.default_rng(42 if model=='resnet50' else 7)
    x = np.arange(1, epochs+1)
    # ResNet50: bắt đầu cao, hội tụ chậm nhưng đạt accuracy cao
    if model == 'resnet50':
        val_acc = 60 + 30 * (1 - np.exp(-x / 15)) + rng.normal(0, 0.8, epochs)
        train_acc = val_acc + 3 + rng.normal(0, 0.5, epochs)
    else:
        # MobileNetV3: hội tụ nhanh hơn nhờ OneCycleLR
        val_acc = 55 + 28 * (1 - np.exp(-x / 10)) + rng.normal(0, 1.0, epochs)
        train_acc = val_acc + 2 + rng.normal(0, 0.5, epochs)
    train_acc = np.clip(train_acc, 0, 99)
    val_acc = np.clip(val_acc, 0, 99)
    train_loss = 1.6 * np.exp(-x / 12) + 0.2 + rng.normal(0, 0.01, epochs)
    val_loss   = 1.8 * np.exp(-x / 15) + 0.25 + rng.normal(0, 0.02, epochs)
    return pd.DataFrame({
        'epoch': x,
        'train_loss': train_loss,
        'val_loss':   val_loss,
        'train_acc':  train_acc,
        'val_acc':    val_acc,
    })

if df_resnet is None:
    print('\n[Demo] Dùng dữ liệu giả lập cho ResNet50')
    df_resnet = make_demo_history(50, 'resnet50')

if df_mobile is None:
    print('[Demo] Dùng dữ liệu giả lập cho MobileNetV3')
    df_mobile = make_demo_history(50, 'mobilenetv3')

print(f'\nResNet50:\n{df_resnet.describe().round(3)}')
print(f'\nMobileNetV3:\n{df_mobile.describe().round(3)}')

## 2. Loss Curves — ResNet50

In [ ]:
def plot_loss_acc(df: pd.DataFrame, title: str, save_prefix: str):
    """Vẽ 2 subplot: Loss và Accuracy."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = df['epoch']

    # Loss
    ax1.plot(epochs, df['train_loss'], 'b-o', markersize=3, linewidth=1.5, label='Train Loss')
    ax1.plot(epochs, df['val_loss'],   'r-o', markersize=3, linewidth=1.5, label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{title} — Loss')
    ax1.legend()
    ax1.set_xlim(1, len(epochs))

    # Accuracy
    ax2.plot(epochs, df['train_acc'], 'b-o', markersize=3, linewidth=1.5, label='Train Acc')
    ax2.plot(epochs, df['val_acc'],   'r-o', markersize=3, linewidth=1.5, label='Val Acc')
    ax2.axhline(85, color='green', linestyle='--', alpha=0.6, linewidth=1, label='Mục tiêu 85%')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title(f'{title} — Accuracy')
    ax2.legend()
    ax2.set_ylim(0, 105)
    ax2.set_xlim(1, len(epochs))

    # Đánh dấu best epoch
    best_epoch = df['val_acc'].idxmax() + 1
    best_acc = df['val_acc'].max()
    ax2.axvline(best_epoch, color='orange', linestyle=':', alpha=0.8)
    ax2.annotate(f'Best\n{best_acc:.1f}%',
                 xy=(best_epoch, best_acc),
                 xytext=(best_epoch + 2, best_acc - 5),
                 arrowprops=dict(arrowstyle='->', color='orange'),
                 fontsize=9, color='orange')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()

    save_path = FIGURES_DIR / f'{save_prefix}_loss_acc.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f'[Saved] {save_path}')
    plt.show()


plot_loss_acc(df_resnet, 'ResNet50 Transfer Learning', 'fig_4_1_resnet50')

## 3. Loss Curves — MobileNetV3-Large

In [ ]:
plot_loss_acc(df_mobile, 'MobileNetV3-Large', 'fig_4_2_mobilenetv3')

## 4. So sánh 2 mô hình — Val Accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Val Accuracy comparison
ax = axes[0]
ax.plot(df_resnet['epoch'], df_resnet['val_acc'],
        'b-o', markersize=3, linewidth=2, label='ResNet50 (~25M params)')
ax.plot(df_mobile['epoch'], df_mobile['val_acc'],
        'r-o', markersize=3, linewidth=2, label='MobileNetV3-L (~5.5M params)')
ax.axhline(85, color='green', linestyle='--', alpha=0.6, label='Mục tiêu 85%')
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Accuracy (%)')
ax.set_title('So sánh Val Accuracy')
ax.legend()
ax.set_ylim(50, 105)

# Val Loss comparison
ax = axes[1]
ax.plot(df_resnet['epoch'], df_resnet['val_loss'],
        'b-o', markersize=3, linewidth=2, label='ResNet50')
ax.plot(df_mobile['epoch'], df_mobile['val_loss'],
        'r-o', markersize=3, linewidth=2, label='MobileNetV3-L')
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Loss')
ax.set_title('So sánh Val Loss')
ax.legend()

plt.suptitle('So sánh ResNet50 vs MobileNetV3-Large', fontsize=14, fontweight='bold')
plt.tight_layout()
save_path = FIGURES_DIR / 'fig_4_3_model_comparison.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'[Saved] {save_path}')
plt.show()

## 5. Bảng tóm tắt kết quả

In [ ]:
def summarize_training(df: pd.DataFrame, model_name: str) -> dict:
    return {
        'Model': model_name,
        'Best Val Acc (%)': round(df['val_acc'].max(), 2),
        'Best Epoch': df['val_acc'].idxmax() + 1,
        'Final Val Acc (%)': round(df['val_acc'].iloc[-1], 2),
        'Final Val Loss': round(df['val_loss'].iloc[-1], 4),
        'Min Val Loss': round(df['val_loss'].min(), 4),
        'Convergence (epoch 90% max)': (
            df[df['val_acc'] >= df['val_acc'].max() * 0.90]['epoch'].iloc[0]
            if not df[df['val_acc'] >= df['val_acc'].max() * 0.90].empty
            else df['epoch'].iloc[-1]
        ),
    }

summary_df = pd.DataFrame([
    summarize_training(df_resnet, 'ResNet50'),
    summarize_training(df_mobile, 'MobileNetV3-L'),
])

print('\n📊 Bảng so sánh kết quả huấn luyện:')
print(summary_df.to_string(index=False))

# Ghi ra CSV
summary_df.to_csv(FIGURES_DIR.parent / 'results' / 'training_summary.csv', index=False)
print(f'\n[Saved] results/training_summary.csv')

## 6. Phân tích giai đoạn Warm-up vs Fine-tune (ResNet50)

In [ ]:
# Nếu có cột 'phase' trong log của ResNet50
if 'phase' in df_resnet.columns:
    warmup_df = df_resnet[df_resnet['phase'] == 'warmup']
    finetune_df = df_resnet[df_resnet['phase'] == 'finetune']

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(warmup_df['epoch'], warmup_df['val_acc'],
            'b-o', markersize=4, linewidth=2, label='Val Acc (Warmup — Freeze backbone)')
    ax.plot(finetune_df['epoch'], finetune_df['val_acc'],
            'r-o', markersize=4, linewidth=2, label='Val Acc (Fine-tune — Unfreeze all)')

    # Đường phân cách warmup/finetune
    if not warmup_df.empty:
        split_epoch = warmup_df['epoch'].max()
        ax.axvline(split_epoch + 0.5, color='orange', linestyle='--', alpha=0.7, linewidth=2)
        ax.text(split_epoch + 1, ax.get_ylim()[0] + 2, 'Unfreeze↑', color='orange', fontsize=9)

    ax.axhline(85, color='green', linestyle='--', alpha=0.5, label='Mục tiêu 85%')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val Accuracy (%)')
    ax.set_title('ResNet50 — Warm-up vs Fine-tune phase')
    ax.legend()
    ax.set_ylim(40, 105)
    plt.tight_layout()
    save_path = FIGURES_DIR / 'fig_4_4_resnet50_phases.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f'[Saved] {save_path}')
    plt.show()
else:
    print('[INFO] Không có cột phase trong log — đây là dữ liệu demo.')
    print('[INFO] Khi train thực, cột phase sẽ được ResNet50 logger ghi tự động.')

## 7. Export tất cả figure

In [ ]:
figures = list(FIGURES_DIR.glob('fig_4_*.png'))
print(f'\n📁 Các figure đã tạo trong {FIGURES_DIR}:')
for f in sorted(figures):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:45s} ({size_kb:.1f} KB)')

print(f'\n✅ Notebook hoàn thành! Tổng {len(figures)} figure được tạo.')